In [ ]:
# Import of dataset from huggingface
from datasets import load_dataset
import datasets

# Suppress info/debug logging, unnecessary output
datasets.logging.set_verbosity_error()

mnlp_dataset = load_dataset("sapienzanlp-course-materials/hw-mnlp-2026")
# Extract train and test splits
train_data = mnlp_dataset["train"]
test_data = mnlp_dataset["test"]
blind_data = mnlp_dataset["blind"]

In [ ]:
# Visualize dataset structure
print("Dataset structure:")
print(mnlp_dataset)

print("\nSample data:")
print(f"{mnlp_dataset['train'][0]}")

## Task [B1]  
Implement the two baseline semantic search systems:  
- distilbert-base-uncased
- all-MiniLM-L6-v2

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.modules import Transformer, Pooling

# Load DistilBERT using SentenceTransformer's Transformer module
transformer = Transformer("distilbert-base-uncased")
pooling = Pooling(transformer.auto_model.config.hidden_size, pooling_mode="mean")

model_bert = SentenceTransformer(modules=[transformer, pooling])

# Load MiniLM using SentenceTransformer's built-in support for pre-trained models
model_mini = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Cosine similarity search utility function
def cosine_similarity_search(query_embedding, chunk_embeddings, top_k=3):
    """Find top-k chunks most similar to query using cosine similarity"""
    similarities = cosine_similarity(query_embedding.reshape(1, -1), chunk_embeddings).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return top_indices, similarities[top_indices]

Implementation of Hit@k metric as:  
$$Hit@k = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1} [r_i \leq k]$$

In [ ]:

def hit_at_k(retrieved_indices, ground_truth_indices, k):
    """Hit@k: 1 if any ground truth in top-k results, else 0"""
    top_k_results = retrieved_indices[:k]
    gt_set = set(ground_truth_indices)
    has_hit = any(item in gt_set for item in top_k_results)
    return 1 if has_hit else 0

## [A2]
Report additional performance metrics beyond Hit@k

Implementation of MRR@k metric as:  
$$MRR@k = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{rank_i} \cdot \mathbb{1}[rank_i \leq k]$$

In [ ]:
def mrr_at_k(retrieved_indices, ground_truth_indices, k):
    """MRR@k: Reciprocal rank of first k relevant items"""

    relevant_items = set(ground_truth_indices)
    
    # Consider only first k elements
    top_k_predictions = retrieved_indices[:k]
    
    for rank, item_id in enumerate(top_k_predictions, start=1):
        if item_id in relevant_items:
            # Found a relevant item at this rank, return reciprocal
            return 1.0 / rank
            
    # No relevant item found in top-k
    return 0.0

In [ ]:
# Function to evaluate retrieval using both Hit@k and MRR@k metrics
def evaluate_retrieval(queries_embeddings, chunk_embeddings, ground_truths, k_values=[1, 3, 5]):
    results = {}
    for k in k_values:
        hits = [hit_at_k(cosine_similarity_search(q, chunk_embeddings, k)[0], ground_truths[i], k) 
                for i, q in enumerate(queries_embeddings)]
        mrrs = [mrr_at_k(cosine_similarity_search(q, chunk_embeddings, k)[0], ground_truths[i], k) 
                for i, q in enumerate(queries_embeddings)]
        results[f'Hit@{k}'] = np.mean(hits)
        results[f'MRR@{k}'] = np.mean(mrrs)
    return results

In [ ]:
# Helper function for evaluating on real dataset
def evaluate_model_on_dataset(model, dataset, model_name="Model"):
    """Evaluate model on real dataset using Hit@k and MRR@k metrics"""
    print(f"Evaluating {model_name} on {len(dataset)} test samples...")
    
    hits_per_k = {k: [] for k in [1, 3, 5]}
    mrrs_per_k = {k: [] for k in [1, 3, 5]}
    
    for idx, example in enumerate(dataset):
        query = example['query']
        candidates = example['candidate_chunks']
        answer_pos = example['answer_pos']
        
        # Encode query and candidates
        query_emb = model.encode(query, convert_to_numpy=True)
        candidates_emb = model.encode(candidates, convert_to_numpy=True)
        
        # Compute similarities and ranking
        similarities = cosine_similarity(query_emb.reshape(1, -1), candidates_emb).flatten()
        ranked_indices = np.argsort(similarities)[::-1]
        
        # Use hit_at_k and mrr_at_k functions for evaluation
        for k in [1, 3, 5]:
            hits_per_k[k].append(hit_at_k(ranked_indices, [answer_pos], k))
            mrrs_per_k[k].append(mrr_at_k(ranked_indices, [answer_pos], k))
        
        if (idx + 1) % max(1, len(dataset) // 5) == 0:
            print(f"  Progress: {(idx + 1) / len(dataset) * 100:.1f}%")
    
    # Calculate average metrics
    results = {}
    for k in [1, 3, 5]:
        results[f'Hit@{k}'] = np.mean(hits_per_k[k])
        results[f'MRR@{k}'] = np.mean(mrrs_per_k[k])
    
    print(f"Evaluation completed")
    return results

In [ ]:
# Evaluate Baseline 1: DistilBERT
print("="*70)
print("BASELINE 1: DistilBERT")
print("="*70 + "\n")
bert_results = evaluate_model_on_dataset(model_bert, test_data, "DistilBERT")
print("DistilBERT Results:")
for metric in sorted(bert_results.keys()):
    print(f"  {metric}: {bert_results[metric]:.4f}")

In [ ]:
# Evaluate Baseline 2: all-MiniLM-L6-v2
print("="*70)
print("BASELINE 2: all-MiniLM-L6-v2")
print("="*70 + "\n")
mini_results = evaluate_model_on_dataset(model_mini, test_data, "all-MiniLM-L6-v2")
print("all-MiniLM-L6-v2 Results:")
for metric in sorted(mini_results.keys()):
    print(f"  {metric}: {mini_results[metric]:.4f}")

## Task [B2]
Fine-tune DistilBERT using supervised triplet loss to improve semantic search performance. 

Fine-tuning strategy:
1) Create triplet dataset: (query, positive_chunk, negative_chunk)
2) Train using triplet loss to minimize distance between query and correct answer
3) Maximize distance between query and random other chunks
4) Evaluate on test set and compare against baseline

In [ ]:
# Create triplet dataset for fine-tuning
import random
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import TripletMarginLoss

print("="*70)
print("PREPARING TRIPLET DATASET")
print("="*70 + "\n")

class TripletDataset:
    """Dataset that generates triplets: (anchor, positive, negative)"""
    def __init__(self, dataset):
        self.triplets = []
        for example in dataset:
            query = example['query']
            candidates = example['candidate_chunks']
            answer_pos = example['answer_pos']
            positive_chunk = candidates[answer_pos]
            negative_positions = [i for i in range(len(candidates)) if i != answer_pos]
            negative_pos = random.choice(negative_positions) # Randomly select one negative chunk
            negative_chunk = candidates[negative_pos]
            self.triplets.append((query, positive_chunk, negative_chunk))
        
        print(f"Created {len(self.triplets)} triplets from {len(dataset)} samples")
    
    def __len__(self):
        return len(self.triplets)
    
    def __getitem__(self, idx):
        return self.triplets[idx]

def triplet_collate_fn(batch):
    """Extract text triplets from batch"""
    queries = [item[0] for item in batch]
    positives = [item[1] for item in batch]
    negatives = [item[2] for item in batch]
    return queries, positives, negatives

# Prepare triplets data from train dataset
triplet_dataset = TripletDataset(train_data)
train_dataloader = DataLoader(triplet_dataset, shuffle=True, batch_size=16, collate_fn=triplet_collate_fn)
print(f"DataLoader created: {len(train_dataloader)} batches\n")

In [ ]:
# Train the model with Triplet Loss
from sentence_transformers import losses, InputExample

print("="*70)
print("FINE-TUNING WITH TRIPLET LOSS")
print("="*70 + "\n")

# Create a copy of the model for fine-tuning
from copy import deepcopy
finetuned_model = deepcopy(model_bert)

# Convert triplet data to InputExample format for SentenceTransformer
train_examples = []
for anchor, positive, negative in triplet_dataset.triplets:
    train_examples.append(InputExample(texts=[anchor, positive, negative], label=0))

print(f"Created {len(train_examples)} training examples")

# Setup training parameters
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.TripletLoss(model=finetuned_model, triplet_margin=0.5)

print(f"Starting fine-tuning...")
print(f"  - Loss: TripletLoss (margin=0.5)")
print(f"  - Epochs: 3")
print(f"  - Batch size: 16")
print(f"  - Total steps: {len(train_dataloader) * 3}\n")

# Fine-tune the model
warmup_steps = int(len(train_dataloader) * 3 * 0.1)
finetuned_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=warmup_steps,
    show_progress_bar=True
)

print(f"Fine-tuning completed")

In [ ]:
# Evaluate fine-tuned model on the same test data
print("="*70)
print("EVALUATING FINE-TUNED MODEL")
print("="*70 + "\n")

finetuned_evaluation = evaluate_model_on_dataset(finetuned_model, test_data, "Fine-tuned DistilBERT")
print("Fine-tuned Model Results (Hit and MRR):")
for metric in sorted(finetuned_evaluation.keys()):
    print(f"  {metric}: {finetuned_evaluation[metric]:.4f}")

In [ ]:
# Compare Fine-tuned Model vs DistilBERT Baseline vs MiniLM Baseline
print("="*70)
print("RESULTS COMPARISON: Hit@k and MRR@k Metrics")
print("="*70 + "\n")

comparison_data = {
    'Metric': [],
    'DistilBERT (Baseline 1)': [],
    'MiniLM (Baseline 2)': [],
    'Fine-tuned DistilBERT': [],
    'Improvement vs DistilBERT (%)': [],
    'Improvement vs MiniLM (%)': []
}

# Compare Hit@k metrics
print("HIT@K METRICS (Recall-based):")
print("-" * 70)
for metric in ['Hit@1', 'Hit@3', 'Hit@5']:
    baseline_val = bert_results[metric]
    baseline2_val = mini_results[metric]
    finetuned_val = finetuned_evaluation[metric]
    improvement_vs_1 = ((finetuned_val - baseline_val) / baseline_val) * 100
    improvement_vs_2 = ((finetuned_val - baseline2_val) / baseline2_val) * 100
    
    comparison_data['Metric'].append(metric)
    comparison_data['DistilBERT (Baseline 1)'].append(baseline_val)
    comparison_data['MiniLM (Baseline 2)'].append(baseline2_val)
    comparison_data['Fine-tuned DistilBERT'].append(finetuned_val)
    comparison_data['Improvement vs DistilBERT (%)'].append(improvement_vs_1)
    comparison_data['Improvement vs MiniLM (%)'].append(improvement_vs_2)
    
    print(f"{metric}:")
    print(f"  DistilBERT (Baseline):     {baseline_val:.4f}")
    print(f"  MiniLM (Baseline):         {baseline2_val:.4f}")
    print(f"  Fine-tuned DistilBERT:   {finetuned_val:.4f}")
    print(f"  Improvement vs DistilBERT: {improvement_vs_1:+.2f}%")
    print(f"  Improvement vs MiniLM:     {improvement_vs_2:+.2f}%\n")

# Compare MRR@k metrics
print("MRR@K METRICS (Ranking quality):")
print("-" * 70)
for metric in ['MRR@1', 'MRR@3', 'MRR@5']:
    baseline_val = bert_results[metric]
    baseline2_val = mini_results[metric]
    finetuned_val = finetuned_evaluation[metric]
    improvement_vs_1 = ((finetuned_val - baseline_val) / baseline_val) * 100 if baseline_val > 0 else 0
    improvement_vs_2 = ((finetuned_val - baseline2_val) / baseline2_val) * 100 if baseline2_val > 0 else 0
    
    print(f"{metric}:")
    print(f"  DistilBERT (Baseline):     {baseline_val:.4f}")
    print(f"  MiniLM (Baseline):         {baseline2_val:.4f}")
    print(f"  Fine-tuned DistilBERT:   {finetuned_val:.4f}")
    print(f"  Improvement vs DistilBERT: {improvement_vs_1:+.2f}%")
    print(f"  Improvement vs MiniLM:     {improvement_vs_2:+.2f}%\n")

In [ ]:
# Visualize Comparison Results: Three-Way Comparison
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Data preparation
metrics = ['Hit@1', 'Hit@3', 'Hit@5']
distilbert_scores = [comparison_data['DistilBERT (Baseline 1)'][i] for i in range(3)]
minilm_scores = [comparison_data['MiniLM (Baseline 2)'][i] for i in range(3)]
finetuned_scores = [comparison_data['Fine-tuned DistilBERT'][i] for i in range(3)]

# Plot 1: Three-way side-by-side comparison
x = np.arange(len(metrics))
width = 0.25

axes[0, 0].bar(x - width, distilbert_scores, width, label='DistilBERT (Baseline)', color='steelblue', alpha=0.8)
axes[0, 0].bar(x, minilm_scores, width, label='MiniLM (Baseline)', color='skyblue', alpha=0.8)
axes[0, 0].bar(x + width, finetuned_scores, width, label='Fine-tuned DistilBERT', color='coral', alpha=0.8)
axes[0, 0].set_ylabel('Hit@k Score', fontsize=12)
axes[0, 0].set_title('Three-Way Model Comparison', fontweight='bold', fontsize=13)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics)
axes[0, 0].legend(fontsize=10)
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(axis='y', alpha=0.3)
for i, (d, m, f) in enumerate(zip(distilbert_scores, minilm_scores, finetuned_scores)):
    axes[0, 0].text(i - width, d + 0.02, f'{d:.3f}', ha='center', fontsize=8)
    axes[0, 0].text(i, m + 0.02, f'{m:.3f}', ha='center', fontsize=8)
    axes[0, 0].text(i + width, f + 0.02, f'{f:.3f}', ha='center', fontsize=8)

# Plot 2: Improvement vs DistilBERT
improvements_vs_db = comparison_data['Improvement vs DistilBERT (%)']
colors = ['green' if x > 0 else 'red' for x in improvements_vs_db]
axes[0, 1].bar(metrics, improvements_vs_db, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('Improvement (%)', fontsize=12)
axes[0, 1].set_title('Fine-tuned DistilBERT vs DistilBERT Baseline', fontweight='bold', fontsize=13)
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(improvements_vs_db):
    axes[0, 1].text(i, v + 1, f'{v:+.2f}%', ha='center', fontweight='bold', fontsize=10)

# Plot 3: Improvement vs MiniLM
improvements_vs_ml = comparison_data['Improvement vs MiniLM (%)']
colors = ['green' if x > 0 else 'red' for x in improvements_vs_ml]
axes[1, 0].bar(metrics, improvements_vs_ml, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 0].set_ylabel('Improvement (%)', fontsize=12)
axes[1, 0].set_title('Fine-tuned DistilBERT vs MiniLM Baseline', fontweight='bold', fontsize=13)
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(improvements_vs_ml):
    axes[1, 0].text(i, v + 1, f'{v:+.2f}%', ha='center', fontweight='bold', fontsize=10)

# Plot 4: Summary table
summary_text = "DETAILED COMPARISON SUMMARY\n" + "="*60 + "\n\n"
summary_text += f"Models:      DistilBERT, MiniLM-L6-v2, Fine-tuned DistilBERT\n"
summary_text += f"Metrics:     Hit@k (Recall) + MRR@k (Ranking)\n"
summary_text += f"Fine-tuning: DistilBERT + TripletLoss (3 epochs)\n\n"
summary_text += f"Test Dataset:  {len(test_data)} samples\n"
summary_text += f"Training Data: {len(train_data)} triplets\n"
summary_text += f"Loss:          TripletMarginLoss (margin=0.5)\n\n"
summary_text += "="*60 + "\n\n"

# HIT@K SECTION
summary_text += "HIT@K RESULTS:\n"
hit_k_keys = ['Hit@1', 'Hit@3', 'Hit@5']
for i, metric in enumerate(hit_k_keys):
    summary_text += f"  {metric}:\n"
    summary_text += f"    DistilBERT:  {comparison_data['DistilBERT (Baseline 1)'][i]:.4f}\n"
    summary_text += f"    MiniLM:      {comparison_data['MiniLM (Baseline 2)'][i]:.4f}\n"
    summary_text += f"    Fine-tuned:  {comparison_data['Fine-tuned DistilBERT'][i]:.4f}\n"
    summary_text += f"    Δ vs DB:     {comparison_data['Improvement vs DistilBERT (%)'][i]:+.2f}%\n"
    summary_text += f"    Δ vs ML:     {comparison_data['Improvement vs MiniLM (%)'][i]:+.2f}%\n\n"

# MRR@K SECTION
summary_text += "MRR@K RESULTS:\n"
for metric in ['MRR@1', 'MRR@3', 'MRR@5']:
    db_val = bert_results[metric]
    ml_val = mini_results[metric]
    ft_val = finetuned_evaluation[metric]
    summary_text += f"  {metric}:\n"
    summary_text += f"    DistilBERT:  {db_val:.4f}\n"
    summary_text += f"    MiniLM:      {ml_val:.4f}\n"
    summary_text += f"    Fine-tuned:  {ft_val:.4f}\n\n"

axes[1, 1].text(0.02, 0.95, summary_text, transform=axes[1, 1].transAxes, 
                fontsize=8, verticalalignment='top', family='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8, edgecolor='gray'))
axes[1, 1].axis('off')


plt.tight_layout()
plt.show()